Flatten `bronze_company_registry` (nested INPI RNE JSON) into on clean row per SIREN

In [0]:
CATALOG = 'credit_risk_project'
BRONZE_SCHEMA = f"{CATALOG}.credit_risk_project_bronze"
SILVER_SCHEMA = f"{CATALOG}.credit_risk_project_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_registry = spark.table(f"{BRONZE_SCHEMA}.bronze_company_registry")

# df_registry.display()

entreprise = "formality.content.personneMorale.identite.entreprise"
description = "formality.content.personneMorale.identite.description"
address = "formality.content.personneMorale.adresseEntreprise.adresse"

df_company = (
    df_registry.select(
        F.col("siren").cast("string").alias("siren"),
        F.col(f"{entreprise}.denomination").alias("company_name"),
        F.col(f"{entreprise}.formeJuridique").alias("forme_juridique"),
        F.col(f"{entreprise}.codeApe").alias("code_ape"),
        F.col("updatedAt").alias("bronze_updated_at"),
        F.to_date(F.col(f"{entreprise}.dateImmat")).alias("date_immatriculation"),
        F.to_date(F.col(f"{entreprise}.dateDebutActiv")).alias("date_debut_activite"),
        F.col(f"{entreprise}.nombreSalarie").cast("int").alias("nombre_salarie"),
        F.col(f"{description}.montantCapital").cast("double").alias("capital_social_eur"),
        F.col(f"{description}.deviseCapital").alias("devise_capital"),
        F.to_date(F.col(f"{description}.dateClotureExerciceSocial")).alias("date_cloture_exercice"),
        F.col(f"{description}.objet").alias("objet_social"),
        F.col(f"{address}.codePostal").alias("code_postal"),
        F.col(f"{address}.commune").alias("commune"),
        F.col(f"{address}.codeInseeCommune").alias("code_insee_commune"),
        F.concat_ws(
            " ",
            F.col(f"{address}.numVoie"),
            F.col(f"{address}.typeVoie"),
            F.col(f"{address}.voie"),
        ).alias("address_voie"),
        F.col("updatedAt").alias("bronze_update_at"),
        )
    .withColumn(
        "_rn",
        F.row_number().over(
            Window.partitionBy("siren").orderBy(F.col("bronze_updated_at").desc_nulls_last())
        ),
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn("silver_ingested_at", F.current_timestamp())
)

print(f"silver_companies row count: {df_company.count()}")
df_company.display()
    






In [0]:
(
    df_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema",True)
    .saveAsTable(f"{SILVER_SCHEMA}.silver_company")
)

print(f"Saved {SILVER_SCHEMA}.silver_company")


## Sanity check


In [0]:
print("companies:", spark.table(f"{SILVER_SCHEMA}.silver_company").count())
display(spark.table(f"{SILVER_SCHEMA}.silver_company")
        .select("siren", "company_name", "forme_juridique", "capital_social_eur"))